<a href="https://colab.research.google.com/github/HalvorHSteff/Padel_elo/blob/main/padel_elo_kode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
from gspread_dataframe import get_as_dataframe
import gspread
from google.colab import auth
import google.auth

import numpy as np
import pandas as pd
import requests
import os
import csv

In [9]:
#google collab bullshit, bare ignorer det

drive.mount('/content/drive')

auth.authenticate_user()
creds, _ = google.auth.default()

gc = gspread.authorize(creds)

sheet = gc.open("Padel").sheet1
df = get_as_dataframe(sheet)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
#her er bare jeg som lager df basert på google sheets som vi lagrer data i, dette kan gjøres på mange måter

df_padel = df.iloc[:, 3:11]
df_padel.columns = ['Player 1', 'Player 2', 'Game_1', 'Set_1', 'Set_2', 'Game_2', 'Player 3', 'Player 4']
df_data = df_padel.dropna(how='all')

start_idx = df_data.first_valid_index()
end_idx = df_data.last_valid_index()

df_padel_data = df_data.loc[(start_idx + 1):end_idx].reset_index(drop=True)

print(df_padel_data)

   Player 1     Player 2 Game_1 Set_1 Set_2 Game_2     Player 3     Player 4
0    Halvor        Simon      7     1     0      5         Erik      Bjørnar
1    Halvor      Bjørnar      7     1     0      5         Erik        Simon
2    Halvor         Erik      4     0     1      6        Simon      Bjørnar
3    Halvor      Bjørnar      5     0     1      7         Erik        Simon
4    Halvor        Simon      6     1     0      0         Erik      Bjørnar
5    Halvor         Erik      3     0     1      6        Simon      Bjørnar
6    Halvor         Erik      6     0     1      7      Bjørnar  Carl-Henrik
7    Halvor  Carl-Henrik      6     1     0      1         Erik      Bjørnar
8    Halvor      Bjørnar      6     1     0      0         Erik  Carl-Henrik
9    Halvor      Bjørnar      6     1     0      3         Erik  Carl-Henrik
10   Halvor         Erik      3     0     1      6      Bjørnar  Carl-Henrik
11   Halvor  Carl-Henrik      6     1     0      0         Erik      Bjørnar

In [13]:
def elo_endring(elo_lag_1, elo_lag_2, lag_1_faktisk, lag_2_faktisk, k=40):
    #regner ut eloendringen til lag basert på forventet resiltat og faktisk resulat av kampen, k er en variabel du kan sette til det du vil
    #det eneste det endrer er hvor mye elo max kan endre seg hver kamp
    lag_1_forventet, lag_2_forventet = forventet_result(elo_lag_1, elo_lag_2)

    elo_lag_1_endring = k * (lag_1_faktisk - lag_1_forventet)
    elo_lag_2_endring = k * (lag_2_faktisk - lag_2_forventet)

    return elo_lag_1_endring, elo_lag_2_endring

def forventet_result(elo_lag_1, elo_lag_2):
    #regner ut forventet resultat av kampen basert på elo
    we = 1 / (1 + 10 ** ((elo_lag_2 - elo_lag_1) / 400))
    return we, 1 - we



In [14]:
elo={}

for idx,row in df_padel_data.iterrows():

    #finner navnene til spillerene sån riktig elo blir regnet ut
    spiller_1 = row['Player 1']
    spiller_2 = row['Player 2']
    spiller_3 = row['Player 3']
    spiller_4 = row['Player 4']

    #det faktiske resultet av kampen, som skal seinere regnes ut i forholdt til forventet verdi av kampen
    lag_1_faktisk = row['Set_1']
    lag_2_faktisk = row['Set_2']

    #hvor du vil elo skal starte
    base_elo = 1200

    #setter eloen lik eloen til spilleren, eller hvis spilleren ikke finnes settes eloen til base_elo
    for spiller in [spiller_1, spiller_2, spiller_3, spiller_4]:
        elo.setdefault(spiller, base_elo)


    #regner ut snitt elo av laget og bruker den som base for å oppdatere elo til spillerene
    elo_lag_1 = (elo[spiller_1] + elo[spiller_2])/2
    elo_lag_2 = (elo[spiller_3] + elo[spiller_4])/2

    #regner ut endringen i eloen til laget
    elo_lag_1_endring, elo_lag_2_endring = elo_endring(elo_lag_1, elo_lag_2, lag_1_faktisk, lag_2_faktisk)

    #Endrer faktisk eloen til hver spiller i dicten
    elo[spiller_1]= round(elo[spiller_1] + elo_lag_1_endring)
    elo[spiller_2]= round(elo[spiller_2] + elo_lag_1_endring)
    elo[spiller_3]= round(elo[spiller_3] + elo_lag_2_endring)
    elo[spiller_4]= round(elo[spiller_4] + elo_lag_2_endring)

#printer dct med elo til hver spiller
print(elo)

{'Halvor': 1417, 'Simon': 1217, 'Erik': 996, 'Bjørnar': 1262, 'Carl-Henrik': 1193, 'Alf': 1177, 'Peder': 1238, 'Fredrik': 1100}


In [ ]:
#her setter jeg det bare automatisk inn igjen i google sheets
for idx, (player, rating) in enumerate(elo.items()):
    sheet.update_cell(idx + 5, 1, player)
    sheet.update_cell(idx + 5, 2, rating)